# Clase 194 — Versionado de datos con DVC

> Parte 4 · MLOps · Fuente: Huyen cap. 6 + docs DVC 3.x.

**Objetivo**: trackear un dataset con DVC, configurar un remote local, declarar un pipeline reproducible en `dvc.yaml` y correr experimentos sin contaminar `git log`.

> ⚠️ Este notebook ejecuta comandos shell (`!dvc ...`, `!git ...`) en un directorio temporal `/tmp/dvc_demo` (Linux/Mac) o `%TEMP%/dvc_demo` (Windows). Requiere `dvc` instalado: `pip install dvc`.

## Setup

In [ ]:
import os, shutil, subprocess, tempfile, json
from pathlib import Path
import pandas as pd
import seaborn as sns

WORK = Path(tempfile.gettempdir()) / 'dvc_demo'
REMOTE = Path(tempfile.gettempdir()) / 'dvc_remote_demo'
for p in (WORK, REMOTE):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True)
os.chdir(WORK)
print('cwd:', Path.cwd())

In [ ]:
def sh(cmd):
    """Ejecuta comando shell y muestra stdout/stderr. Devuelve returncode."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print('STDERR:', r.stderr)
    return r.returncode

sh('git --version && dvc --version')

## 1. `git init` + `dvc init`

DVC vive *encima* de git: necesita un repo git para guardar los `.dvc` pointers.

In [ ]:
sh('git init -q && git config user.email demo@local && git config user.name demo')
sh('dvc init -q')
sh('git add .dvc .dvcignore && git commit -q -m "init dvc"')
sh('ls -la .dvc/')

## 2. `dvc add` — trackear un dataset

Generamos un CSV pequeño (titanic ≈ 60 KB) y lo trackeamos. El blob va al cache local; en git solo queda un `.dvc` de ~200 B.

In [ ]:
Path('data/raw').mkdir(parents=True, exist_ok=True)
tit = sns.load_dataset('titanic')
tit.to_csv('data/raw/titanic.csv', index=False)
print(f'CSV size: {Path("data/raw/titanic.csv").stat().st_size:,} bytes')

sh('dvc add data/raw/titanic.csv')
sh('cat data/raw/titanic.csv.dvc')   # el pointer
sh('cat data/raw/.gitignore')          # DVC agrega el blob al .gitignore

In [ ]:
# Commit del puntero (NO del blob — el blob ya está en .gitignore)
sh('git add data/raw/titanic.csv.dvc data/raw/.gitignore && git commit -q -m "add titanic dataset"')
sh('git ls-files data/')   # solo .dvc y .gitignore — el CSV NO está en git

## 3. Remote y `dvc push`

Configuramos un remote local (simula S3). En producción sería `dvc remote add -d origin s3://bucket/path`.

In [ ]:
sh(f'dvc remote add -d local {REMOTE}')
sh('git add .dvc/config && git commit -q -m "add remote"')
sh('dvc push')

# El blob aparece en el remote, organizado por hash MD5 (prefijo de 2 chars)
for p in sorted(REMOTE.rglob('*'))[:10]:
    if p.is_file(): print(p.relative_to(REMOTE))

## 4. Pipeline declarativo (`dvc.yaml`)

Tres stages: `prepare → train → evaluate`. Cada uno declara `deps`, `outs` y opcionalmente `params`/`metrics`. `dvc repro` re-ejecuta solo lo que cambió.

In [ ]:
Path('src').mkdir(exist_ok=True)

Path('src/prepare.py').write_text('''\
import pandas as pd, sys
df = pd.read_csv(sys.argv[1]).dropna(subset=["age", "fare", "survived"])
df = df[["age", "fare", "pclass", "sex", "survived"]]
df["sex"] = (df["sex"] == "male").astype(int)
df.to_csv(sys.argv[2], index=False)
''')

Path('src/train.py').write_text('''\
import pandas as pd, yaml, joblib, sys
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
params = yaml.safe_load(open("params.yaml"))["train"]
df = pd.read_csv(sys.argv[1])
X, y = df.drop(columns=["survived"]), df["survived"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=params["test_size"], random_state=42)
m = LogisticRegression(C=params["C"], max_iter=1000).fit(Xtr, ytr)
joblib.dump({"model": m, "Xte": Xte, "yte": yte}, sys.argv[2])
''')

Path('src/evaluate.py').write_text('''\
import joblib, json, sys
from sklearn.metrics import accuracy_score, f1_score
b = joblib.load(sys.argv[1])
pred = b["model"].predict(b["Xte"])
json.dump({"accuracy": accuracy_score(b["yte"], pred), "f1": f1_score(b["yte"], pred)},
          open(sys.argv[2], "w"), indent=2)
''')

Path('params.yaml').write_text('train:\n  test_size: 0.2\n  C: 1.0\n')

Path('dvc.yaml').write_text('''\
stages:
  prepare:
    cmd: python src/prepare.py data/raw/titanic.csv data/processed/clean.csv
    deps:
      - data/raw/titanic.csv
      - src/prepare.py
    outs:
      - data/processed/clean.csv
  train:
    cmd: python src/train.py data/processed/clean.csv model.pkl
    deps:
      - data/processed/clean.csv
      - src/train.py
    params:
      - train.test_size
      - train.C
    outs:
      - model.pkl
  evaluate:
    cmd: python src/evaluate.py model.pkl metrics.json
    deps:
      - model.pkl
      - src/evaluate.py
    metrics:
      - metrics.json:
          cache: false
''')

print('Pipeline declarado.')

In [ ]:
sh('dvc repro')
print('--- metrics.json ---')
print(Path('metrics.json').read_text())
print('--- dvc.lock (primeras líneas) ---')
print('\n'.join(Path('dvc.lock').read_text().splitlines()[:20]))

## 5. Re-ejecución incremental

Cambiamos `C` en `params.yaml`. DVC detecta que cambió un `param` de `train` y re-ejecuta solo `train + evaluate` (no `prepare`).

In [ ]:
import yaml
p = yaml.safe_load(open('params.yaml'))
p['train']['C'] = 0.01
yaml.safe_dump(p, open('params.yaml', 'w'))

sh('dvc repro')   # observá: "Stage 'prepare' didn't change, skipping"
print(Path('metrics.json').read_text())

## 6. Experimentos con `dvc exp run`

Tres corridas variando `C`, sin tocar `git log`.

In [ ]:
sh('git add . && git commit -q -m "pipeline ready"')
for c in [0.01, 1.0, 100.0]:
    sh(f'dvc exp run -S train.C={c} --quiet')
sh('dvc exp show --no-pager --drop ".*" --keep "Experiment|C|accuracy|f1"')

## Ejercicio guiado

1. Agregá un cuarto stage `register` que copie `model.pkl` a `models/<git-sha-corto>.pkl`. Pista: usá `$(git rev-parse --short HEAD)` en el `cmd`.
2. Borrá el cache local (`rm -rf .dvc/cache data/processed model.pkl`) y reconstruí TODO con `dvc pull && dvc repro`. Confirmá que el hash final de `metrics.json` coincide con el de `dvc.lock`.
3. Agregá una imagen `confusion_matrix.png` como `outs` del stage `evaluate` y visualizala con `dvc plots show`.

## Conclusiones

- DVC desacopla **versionado lógico** (git, liviano) de **storage físico** (remote, pesado).
- `dvc.yaml` + `dvc.lock` son el contrato de reproducibilidad: mismo commit → misma corrida.
- `dvc exp` reemplaza el patrón "branch por experimento" que ensucia git.
- En producción real: el remote es S3/GCS, el CI hace `dvc pull` antes de entrenar, y `dvc.lock` se commitea junto al PR.